# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model to detect billiard table pockets, then exports it as a
**TFLite FP16** model ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU.

**Pipeline:**
1. Install dependencies
2. Load dataset from Roboflow (or upload your own)
3. Train YOLOv8n
4. Validate
5. Export to TFLite FP16
6. Smoke-test the TFLite model
7. Download `pocket_detector.tflite`

---
**Single class:** `pocket`  
**Target:** ~300–500 labeled images  
**Output:** `pocket_detector.tflite` (~3 MB)

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 2. Dataset

**Option A (recommended):** Load from Roboflow.  
**Option B:** Upload a local dataset zip in YOLOv8 format.

### Annotation guidelines
- **Class:** `pocket` (one class, index 0)
- Draw **tight** bounding boxes around the dark hole, not the jaw
- Label all visible pockets per frame (typically 3–6)
- Corner pockets are roughly square; side pockets slightly wider than tall
- Cover multiple lighting conditions, table colours, and camera angles

In [ ]:
# ── Option A: Roboflow ───────────────────────────────────────────────────────
USE_ROBOFLOW = True  # Set False to use Option B

ROBOFLOW_API_KEY   = "YOUR_API_KEY_HERE"
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"     # e.g. "hereliesaz"
ROBOFLOW_PROJECT   = "YOUR_PROJECT_NAME" # e.g. "billiard-pockets"
ROBOFLOW_VERSION   = 1

if USE_ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download("yolov8")
    DATASET_YAML = dataset.location + "/data.yaml"
    print(f"Dataset: {DATASET_YAML}")

In [ ]:
# ── Option B: Upload zip ─────────────────────────────────────────────────────
# Skip if USE_ROBOFLOW = True.
#
# Expected zip structure:
#   dataset/
#     data.yaml          (path, train, val, nc: 1, names: ['pocket'])
#     train/images/*.jpg
#     train/labels/*.txt
#     valid/images/*.jpg
#     valid/labels/*.txt

if not USE_ROBOFLOW:
    from google.colab import files
    import zipfile
    print("Upload your dataset zip:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall("/content/dataset")
    DATASET_YAML = "/content/dataset/data.yaml"

In [ ]:
import yaml
with open(DATASET_YAML) as f:
    data = yaml.safe_load(f)
print(yaml.dump(data, default_flow_style=False))
assert 'pocket' in data['names'], f"Expected class 'pocket', got: {data['names']}"
print(f"\u2713 {data['nc']} class(es): {data['names']}")

## 3. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano — ~3 MB after TFLite export

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    name="pocket_detector",
    project="/content/runs",
    # Augmentation tuned for pockets at all angles and lighting
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=45.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

print("Best weights:", results.save_dir)

## 4. Validate

In [ ]:
import os

best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

metrics = trained_model.val(data=DATASET_YAML)

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

if metrics.box.map50 > 0.85:
    print("\n\u2713 mAP50 > 0.85 — good to export")
else:
    print("\n\u26a0 mAP50 below 0.85 — consider more data or more epochs")

## 5. Export to TFLite FP16

In [ ]:
export_path = trained_model.export(
    format="tflite",
    imgsz=640,
    half=True,   # FP16
    int8=False,  # INT8 needs calibration data; use FP16 first
    nms=True,    # bake NMS into the model
)

import os
size_mb = os.path.getsize(export_path) / 1024 / 1024
print(f"TFLite model: {export_path}")
print(f"Size: {size_mb:.1f} MB")

## 6. Smoke-test the TFLite Model

Runs inference on a few validation images to confirm the export is sane.

In [ ]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=export_path)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Input dtype:", input_details[0]['dtype'])

INPUT_SIZE = int(input_details[0]['shape'][1])

def run_tflite(image_path):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (INPUT_SIZE, INPUT_SIZE))
    dtype = input_details[0]['dtype']
    inp = (resized / 255.0).astype(dtype)
    inp = np.expand_dims(inp, axis=0)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])[0]
    return img_rgb, output, w, h

yaml_dir = DATASET_YAML.replace('data.yaml', '')
val_images = glob.glob(yaml_dir + 'valid/images/*.jpg')[:4]
if not val_images:
    val_images = glob.glob(yaml_dir + 'valid/images/*.png')[:4]

fig, axes = plt.subplots(1, max(len(val_images), 1), figsize=(20, 5))
if len(val_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, val_images):
    img_rgb, detections, w, h = run_tflite(img_path)
    img_draw = img_rgb.copy()
    count = 0
    for det in detections:
        if len(det) >= 5 and float(det[4]) > 0.3:
            x1 = int(det[0] * w / INPUT_SIZE)
            y1 = int(det[1] * h / INPUT_SIZE)
            x2 = int(det[2] * w / INPUT_SIZE)
            y2 = int(det[3] * h / INPUT_SIZE)
            cv2.rectangle(img_draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img_draw, f"{float(det[4]):.2f}",
                        (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            count += 1
    ax.imshow(img_draw)
    ax.set_title(f"{count} pockets")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Download

In [ ]:
import shutil
from google.colab import files

OUTPUT_NAME = "pocket_detector.tflite"
shutil.copy(export_path, f"/content/{OUTPUT_NAME}")

size_mb = os.path.getsize(f"/content/{OUTPUT_NAME}") / 1024 / 1024
print(f"{OUTPUT_NAME}  ({size_mb:.1f} MB)")
files.download(f"/content/{OUTPUT_NAME}")

## 8. Next Steps — Android Integration (v1.4)

1. Copy `pocket_detector.tflite` into `app/src/main/assets/`

2. Add to `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.16.1")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
```

3. Implement `PocketDetector`:
```kotlin
class TflitePocketDetector(context: Context) : PocketDetector {
    private val interpreter = Interpreter(
        FileUtil.loadMappedFile(context, "pocket_detector.tflite")
    )
    override fun detect(yBytes: ByteArray, width: Int, height: Int): List<PointF>? {
        // 1. Resize yBytes to 640x640 (grayscale -> RGB by replicating channel)
        // 2. Run interpreter
        // 3. Parse output boxes, return centres with confidence > 0.4
    }
}
```

4. Pass to `TableScanAnalyzer` in `ProtractorScreen.kt`:
```kotlin
val detector = remember { TflitePocketDetector(context) }
val tableScanAnalyzer = remember(tableScanViewModel) {
    TableScanAnalyzer(
        tableScanViewModel::onFrame,
        tableScanViewModel::onFeltColorSampled,
        pocketDetector = detector,
    )
}
```

The Hough fallback stays active whenever `pocketDetector.detect()` returns `null`.